# Advanced: Multi-Strategy Partial Masking with PAMOLA.CORE

**Goal**: Master all 4 partial masking strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Fixed position-based masking (prefix/suffix preservation)
- **Strategy 2**: Pattern-based masking (email, phone, SSN, credit card)
- **Strategy 3**: Random percentage masking
- **Strategy 4**: Word boundary preservation
- Calculate advanced privacy metrics
- Analyze masking effectiveness
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of privacy concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/masking/
        └── 02_partial_masking_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.masking.partial_masking_op import (
        PartialMaskingOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 5 columns)
- Sample records (first 5 rows)
- Column statistics (types, unique counts, sample values)

**Dataset features:**
- 1000 records for comprehensive testing
- Multiple sensitive field types
- Realistic data patterns
- Suitable for all masking strategies

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate realistic email addresses
    first_names = ['john', 'jane', 'alice', 'bob', 'charlie', 'david', 'emma', 'frank']
    last_names = ['smith', 'johnson', 'williams', 'brown', 'jones', 'garcia', 'miller', 'davis']
    domains = ['email.com', 'company.org', 'test.net', 'work.biz', 'example.com']
    
    emails = [
        f"{np.random.choice(first_names)}.{np.random.choice(last_names)}{np.random.randint(1, 999)}@{np.random.choice(domains)}"
        for _ in range(n_records)
    ]
    
    # Generate phone numbers
    phones = [f"555-{np.random.randint(1000, 9999)}" for _ in range(n_records)]
    
    # Generate SSN
    ssns = [
        f"{np.random.randint(100, 999)}-{np.random.randint(10, 99)}-{np.random.randint(1000, 9999)}"
        for _ in range(n_records)
    ]
    
    # Generate credit cards
    card_prefixes = ['4532', '5425', '3782', '6011']
    credit_cards = [
        f"{np.random.choice(card_prefixes)}-{np.random.randint(1000, 9999)}-{np.random.randint(1000, 9999)}-{np.random.randint(1000, 9999)}"
        for _ in range(n_records)
    ]
    
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'email': emails,
        'phone': phones,
        'ssn': ssns,
        'credit_card': credit_cards
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
        print(f"    Sample: {df[col].iloc[0]}")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n💡 Perfect dataset for testing all 4 strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for each strategy
   - `"strategy1_field": "email"` → YOUR column name
   - `"strategy2_field": "phone"` → YOUR column name
   - `"strategy3_field": "email"` → YOUR column name
   - `"strategy4_field": "phone"` → YOUR column name
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Unique value counts per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "email",        # Fixed position masking
    "strategy2_field": "phone",        # Pattern-based masking
    "strategy3_field": "email",          # Random percentage masking
    "strategy4_field": "phone"   # Word boundary masking
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  ✓ {strategy:20s}: '{field_name}' ({df[field_name].nunique()} unique values)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy",
    description="Multi-strategy partial masking processing",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Fixed Position-Based Masking

**How to use:**
- Review pre-configured parameters
- Run to execute mask strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `mask_strategy='fixed'`: Position-based masking
- `unmasked_prefix=3`: Keep first 3 chars visible
- `unmasked_suffix=10`: Keep last 10 chars visible
- `preserve_separators=True`: Keep @, ., - visible
- `mode='ENRICH'`: Keeps original + adds generalized field

**Note:** Creates `{field_name}_fixed` with prefix/suffix preserved

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: FIXED POSITION-BASED MASKING")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Fixed position",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = PartialMaskingOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy1_field"],
    mode='ENRICH',
    
    # Mask strategy selection
    mask_strategy='fixed',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy1_field']}_fixed",
    output_format='csv',
    
    # Position-based parameters
    mask_char='*',
    unmasked_prefix=3,
    unmasked_suffix=10,
    
    # Format preservation
    preserve_separators=True,
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_fixed',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_fixed' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    field_s1 = FIELD_CONFIG["strategy1_field"]
    output_field_s1 = f"{field_s1}_fixed"
    
    print(f"\n📊 Sample masked values:")
    for i in range(min(5, len(result_df_s1))):
        orig = df[field_s1].iloc[i]
        masked = result_df_s1[output_field_s1].iloc[i]
        print(f"  {orig} → {masked}")

## STRATEGY 2: Pattern-Based Masking

**How to use:**
- Review pre-configured parameters
- Run to execute mask_strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `mask_strategy='pattern'`: Uses predefined patterns
- `pattern_type='phone'`: Phone number pattern
- `preserve_separators=True`: Keep - visible
- `mode='ENRICH'`: Keeps original + adds generalized field

**Note:** Creates `{field_name}_pattern` with pattern-aware masking

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: PATTERN-BASED MASKING")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Pattern-based",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = PartialMaskingOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy2_field"],
    mode='ENRICH',
    
    # Mask strategy selection
    mask_strategy='pattern',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy2_field']}_pattern",
    output_format='csv',
    
    # Pattern parameters
    pattern_type='phone',
    mask_char='*',
    preserve_separators=True,
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_pattern',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results (NEWEST file)
output_files_s2 = sorted(
    list((task_dir / 'strategy2_pattern' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    field_s2 = FIELD_CONFIG["strategy2_field"]
    output_field_s2 = f"{field_s2}_pattern"
    
    print(f"\n📊 Sample masked values:")
    for i in range(min(5, len(result_df_s2))):
        orig = result_df_s1[field_s2].iloc[i]
        masked = result_df_s2[output_field_s2].iloc[i]
        print(f"  {orig} → {masked}")

## STRATEGY 3: Random Percentage Masking

**How to use:**
- Review pre-configured parameters
- Run to execute mask_strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `mask_strategy='random'`: Random character masking
- `mask_percentage=50`: Mask 50% of characters
- `preserve_separators=True`: Keep - visible
- `mode='ENRICH'`: Keeps original + adds generalized field

**Note:** Creates `{field_name}_random` with 50% random masking

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: RANDOM PERCENTAGE MASKING")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Random percentage",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = PartialMaskingOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy3_field"],
    mode='ENRICH',
    
    # Mask strategy selection
    mask_strategy='random',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy3_field']}_random",
    output_format='csv',
    
    # Random masking parameters
    mask_char='*',
    mask_percentage=50,
    preserve_separators=True,
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_random',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results (NEWEST file)
output_files_s3 = sorted(
    list((task_dir / 'strategy3_random' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    field_s3 = FIELD_CONFIG["strategy3_field"]
    output_field_s3 = f"{field_s3}_random"
    
    print(f"\n📊 Sample masked values:")
    for i in range(min(5, len(result_df_s3))):
        orig = result_df_s2[field_s3].iloc[i]
        masked = result_df_s3[output_field_s3].iloc[i]
        print(f"  {orig} → {masked}")

## STRATEGY 4: Word Boundary Preservation

**How to use:**
- Review pre-configured parameters
- Run to execute mask strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `mask_strategy='words'`: Word-based masking
- `preserve_word_boundaries=True`: Maintain word structure
- `preserve_separators=True`: Keep - visible
- `mode='ENRICH'`: Keeps original + adds generalized field

**Note:** Creates `{field_name}_words` with word boundaries preserved

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 4: WORD BOUNDARY PRESERVATION")
print("=" * 80 + "\n")

tracker_s4 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 4: Word boundaries",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s4 = PartialMaskingOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy4_field"],
    mode='ENRICH',
    
    # Mask strategy selection
    mask_strategy='words',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy4_field']}_words",
    output_format='csv',
    
    # Word boundary parameters
    mask_char='*',
    preserve_word_boundaries=True,
    preserve_separators=True,
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 4 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s4 = operation_s4.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy4_words',
    reporter=task_reporter,
    progress_tracker=tracker_s4,
    **kwargs
)

elapsed_s4 = time.time() - start_time
print(f"\n✅ Strategy 4 completed in {elapsed_s4:.2f}s")

# Load results (NEWEST file)
output_files_s4 = sorted(
    list((task_dir / 'strategy4_words' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s4:
    result_df_s4 = pd.read_csv(output_files_s4[0])
    field_s4 = FIELD_CONFIG["strategy4_field"]
    output_field_s4 = f"{field_s4}_words"
    
    print(f"\n📊 Sample masked values:")
    for i in range(min(5, len(result_df_s4))):
        orig = result_df_s3[field_s4].iloc[i]
        masked = result_df_s4[output_field_s4].iloc[i]
        print(f"  {orig} → {masked}")

## Step 5: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Masking characteristics per mask strategy

**Note:** Fastest mask strategy isn't always best - consider data quality

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Fixed):      {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Pattern):    {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Random):     {elapsed_s3:6.2f}s")
print(f"  Strategy 4 (Words):      {elapsed_s4:6.2f}s")
print(f"  Total:                   {elapsed_s1 + elapsed_s2 + elapsed_s3 + elapsed_s4:6.2f}s")

print(f"\n📈 Masking Characteristics:")
print(f"  Strategy 1: Preserves prefix/suffix (predictable)")
print(f"  Strategy 2: Pattern-aware (domain-specific)")
print(f"  Strategy 3: Random positioning (unpredictable)")
print(f"  Strategy 4: Word structure maintained (readable)")

## Step 6: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Focus on validation points per mask strategy

**What you'll see (per mask strategy):**
1. **Metrics JSON**: Mask rate, visibility %, mask strategy performance
2. **Visualizations**: Charts displayed inline (latest batch only)

**Validate:**
- ✅ Mask rate 30-70% (balanced privacy/utility)
- ✅ All records processed successfully
- ✅ No errors in execution

**Note:** Only newest files shown. Multiple test runs create versions - older artifacts excluded automatically

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# Review all strategies
strategy_dirs = [
    ('strategy1_fixed', 'Strategy 1: Fixed Position'),
    ('strategy2_pattern', 'Strategy 2: Pattern-Based'),
    ('strategy3_random', 'Strategy 3: Random Percentage'),
    ('strategy4_words', 'Strategy 4: Word Boundaries')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. METRICS JSON (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Display key metrics
                if 'partial_mask_rate' in metrics:
                    print(f"   Mask Rate:        {metrics['partial_mask_rate']:.2%}")
                if 'average_visibility' in metrics:
                    print(f"   Avg Visibility:   {metrics['average_visibility']:.2%}")
                if 'values_masked' in metrics:
                    print(f"   Values Masked:    {metrics['values_masked']}")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. VISUALIZATIONS (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## Step 7: Calculate Privacy Metrics

**How to use:**
- Run AFTER all strategies complete
- Calculates masking effectiveness

**What you'll see:**
- Original data: unique values per field
- Masked data: masking rates per mask strategy
- Average character masking percentages

**Note:** Requires result_df_s{N} from mask strategy execution cells

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS")
print("=" * 80 + "\n")

def calculate_masking_stats(original_series, masked_series):
    """Calculate masking statistics."""
    total_chars = original_series.str.len().sum()
    masked_chars = masked_series.apply(lambda x: str(x).count('*')).sum()
    mask_percentage = (masked_chars / total_chars * 100) if total_chars > 0 else 0
    
    return {
        'total_chars': total_chars,
        'masked_chars': masked_chars,
        'mask_percentage': mask_percentage
    }

# Check if strategies completed
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    field_s3 = FIELD_CONFIG["strategy3_field"]
    field_s4 = FIELD_CONFIG["strategy4_field"]
    
    print(f"📊 ORIGINAL DATA:")
    print(f"   {field_s1}: {df[field_s1].nunique()} unique values")
    print(f"   {field_s2}: {df[field_s2].nunique()} unique values")
    print(f"   {field_s3}: {df[field_s3].nunique()} unique values")
    print(f"   {field_s4}: {df[field_s4].nunique()} unique values")
    
    print(f"\n✨ MASKING EFFECTIVENESS:")
    
    # Strategy 1
    if 'result_df_s1' in locals() and result_df_s1 is not None:
        stats_s1 = calculate_masking_stats(df[field_s1], result_df_s1[f"{field_s1}_fixed"])
        print(f"\n  Strategy 1 (Fixed):")
        print(f"    Characters masked: {stats_s1['mask_percentage']:.1f}%")
    
    # Strategy 2
    if 'result_df_s2' in locals() and result_df_s2 is not None:
        stats_s2 = calculate_masking_stats(result_df_s1[field_s2], result_df_s2[f"{field_s2}_pattern"])
        print(f"\n  Strategy 2 (Pattern):")
        print(f"    Characters masked: {stats_s2['mask_percentage']:.1f}%")
    
    # Strategy 3
    if 'result_df_s3' in locals() and result_df_s3 is not None:
        stats_s3 = calculate_masking_stats(result_df_s2[field_s3], result_df_s3[f"{field_s3}_random"])
        print(f"\n  Strategy 3 (Random):")
        print(f"    Characters masked: {stats_s3['mask_percentage']:.1f}%")
    
    # Strategy 4
    if 'result_df_s4' in locals() and result_df_s4 is not None:
        stats_s4 = calculate_masking_stats(result_df_s3[field_s4], result_df_s4[f"{field_s4}_words"])
        print(f"\n  Strategy 4 (Words):")
        print(f"    Characters masked: {stats_s4['mask_percentage']:.1f}%")
        
except NameError:
    print("⚠️  Run Step 3 first!")

## Step 8: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports anonymized datasets for each mask strategy

**What you'll see (per mask strategy):**
- Available columns list
- Export path
- Preview (first 5 rows)
- Statistics (masking %)

**Output structure:**
```
advanced_output/
├── strategy1_fixed/
│   └── fixed_masked_data.csv
├── strategy2_pattern/
│   └── pattern_masked_data.csv
├── strategy3_random/
│   └── random_masked_data.csv
└── strategy4_words/
    └── words_masked_data.csv
```

**Note:** Files include both original and masked fields for comparison

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Get field config
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    field_s3 = FIELD_CONFIG["strategy3_field"]
    field_s4 = FIELD_CONFIG["strategy4_field"]
except NameError:
    print("❌ Run Step 3 first!")
    raise

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Fixed Position Masking
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: FIXED POSITION MASKING")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_fixed'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_fixed"
    
    if output_col_s1 in result_df_s1.columns:
        export_cols_s1 = ['record_id', field_s1, output_col_s1]
        existing_cols_s1 = [col for col in export_cols_s1 if col in result_df_s1.columns]
        
        final_df_s1 = result_df_s1[existing_cols_s1].copy()
        
        output_path_s1 = s1_dir / 'fixed_masked_data.csv'
        try:
            final_df_s1.to_csv(output_path_s1, index=False)
            print(f"\n✅ Saved: {output_path_s1}")
            print(f"   Rows: {len(final_df_s1):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s1.head())
    else:
        print(f"\n❌ Column '{output_col_s1}' not found")
else:
    print("\n⚠️  Strategy 1 data not available")

# ============================================================================
# STRATEGY 2: Pattern-Based Masking
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: PATTERN-BASED MASKING")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_pattern'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_fixed"
    output_col_s2 = f"{field_s2}_pattern"
    
    if output_col_s2 in result_df_s2.columns:
        export_cols_s2 = ['record_id', field_s1, output_col_s1, field_s2, output_col_s2]
        existing_cols_s2 = [col for col in export_cols_s2 if col in result_df_s2.columns]
        
        final_df_s2 = result_df_s2[existing_cols_s2].copy()
        
        output_path_s2 = s2_dir / 'pattern_masked_data.csv'
        try:
            final_df_s2.to_csv(output_path_s2, index=False)
            print(f"\n✅ Saved: {output_path_s2}")
            print(f"   Rows: {len(final_df_s2):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s2.head())
    else:
        print(f"\n❌ Column '{output_col_s2}' not found")
else:
    print("\n⚠️  Strategy 2 data not available")

# ============================================================================
# STRATEGY 3: Random Percentage Masking
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: RANDOM PERCENTAGE MASKING")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_random'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_fixed"
    output_col_s2 = f"{field_s2}_pattern"
    output_col_s3 = f"{field_s3}_random"
    
    if output_col_s3 in result_df_s3.columns:
        export_cols_s3 = ['record_id', field_s1, output_col_s1, 
                          field_s2, output_col_s2, field_s3, output_col_s3]
        existing_cols_s3 = [col for col in export_cols_s3 if col in result_df_s3.columns]
        
        final_df_s3 = result_df_s3[existing_cols_s3].copy()
        
        output_path_s3 = s3_dir / 'random_masked_data.csv'
        try:
            final_df_s3.to_csv(output_path_s3, index=False)
            print(f"\n✅ Saved: {output_path_s3}")
            print(f"   Rows: {len(final_df_s3):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s3.head())
    else:
        print(f"\n❌ Column '{output_col_s3}' not found")
else:
    print("\n⚠️  Strategy 3 data not available")

# ============================================================================
# STRATEGY 4: Word Boundary Preservation
# ============================================================================
if 'result_df_s4' in locals() and result_df_s4 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 4: WORD BOUNDARY PRESERVATION")
    print("=" * 80)
    
    s4_dir = export_base_dir / 'strategy4_words'
    s4_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_fixed"
    output_col_s2 = f"{field_s2}_pattern"
    output_col_s3 = f"{field_s3}_random"
    output_col_s4 = f"{field_s4}_words"
    
    if output_col_s4 in result_df_s4.columns:
        export_cols_s4 = ['record_id', field_s1, output_col_s1,
                          field_s2, output_col_s2, field_s3, output_col_s3,
                          field_s4, output_col_s4]
        existing_cols_s4 = [col for col in export_cols_s4 if col in result_df_s4.columns]
        
        final_df_s4 = result_df_s4[existing_cols_s4].copy()
        
        output_path_s4 = s4_dir / 'words_masked_data.csv'
        try:
            final_df_s4.to_csv(output_path_s4, index=False)
            print(f"\n✅ Saved: {output_path_s4}")
            print(f"   Rows: {len(final_df_s4):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s4.head())
    else:
        print(f"\n❌ Column '{output_col_s4}' not found")
else:
    print("\n⚠️  Strategy 4 data not available")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3 + elapsed_s4
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 4 masking strategies implemented and compared
- ✅ Privacy metrics calculated
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- **Fixed position**: Predictable, preserves structure, best for format-sensitive fields
- **Pattern-based**: Domain-aware, optimal for known patterns (phone, email, SSN)
- **Random percentage**: Unpredictable masking, balanced privacy/utility
- **Word boundaries**: Maintains readability, ideal for text fields

**Strategy recommendations:**
- **Use Fixed** when: Format preservation critical (emails, URLs)
- **Use Pattern** when: Working with standard data types (phone, SSN, credit cards)
- **Use Random** when: Maximum unpredictability needed
- **Use Words** when: Maintaining text structure important

**Next steps:**
- Test with your own datasets
- Tune parameters for optimal results
- Deploy to production environment
- Combine strategies for multi-field masking

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)